In [1]:
import os
import sys

# add the directory containing the notebook to Python path
sys.path.append(os.getcwd())

import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

Problem #5 - A5.10

In [2]:
#helper functions
def sign_pm1(x):
    """Return +/-1 with tie-breaking: +1 if x>=0, else -1."""
    x = np.asarray(x).reshape(-1)
    return np.where(x >= 0, 1.0, -1.0)

def obj_ls(A, b, x):
    r = A @ x - b
    return float(r.T @ r)

def psd_project(S, eps=1e-12):
    """Project symmetric matrix onto PSD cone by eigenvalue clipping."""
    S = 0.5 * (S + S.T)
    w, V = np.linalg.eigh(S)
    w = np.maximum(w, eps)
    return V @ (w[:, None] * V.T)

#generate problem instance
np.random.seed(0)
m, n = 50, 40
A = np.random.randn(m, n)
xhat = sign_pm1(np.random.randn(n))
# noise levels
sigmas = [0.5, 1.0, 2.0, 3.0]

# Precompute gram matrix
Q = A.T @ A


results = []

for s in sigmas:
    b = A @ xhat + s * np.random.randn(m)

    
    x_ls = np.linalg.lstsq(A, b, rcond=None)[0]
    x_a = sign_pm1(x_ls)
    f_a = obj_ls(A, b, x_a)

    
    W = cp.Variable((n + 1, n + 1), PSD=True)
    Z = W[:n, :n]
    z = W[:n, n]

    
    obj = cp.trace(Q @ Z) - 2 * (b @ (A @ z)) + float(b @ b)

    constraints = [
        cp.diag(Z) == 1,
        W[n, n] == 1
    ]

    prob = cp.Problem(cp.Minimize(obj), constraints)
    prob.solve(solver=cp.SCS, verbose=False, eps=1e-5, max_iters=200000)

    if prob.status not in ("optimal", "optimal_inaccurate"):
        raise RuntimeError(f"SDP failed for s={s}, status={prob.status}")

    Z_val = np.array(Z.value)
    z_val = np.array(z.value).reshape(-1)

    
    x_b = sign_pm1(z_val)
    f_b = obj_ls(A, b, x_b)

    
    W_val = np.array(W.value)
    W_val = 0.5 * (W_val + W_val.T)
    evals, evecs = np.linalg.eigh(W_val)
    idx = np.argmax(evals)
    v1 = evecs[:, idx]
    v_part = v1[:n]
    t1 = v1[n]
    x_c = sign_pm1(t1 * v_part)   
    f_c = obj_ls(A, b, x_c)

    
    Sigma = Z_val - np.outer(z_val, z_val)
    Sigma = psd_project(Sigma, eps=1e-10)
    L = np.linalg.cholesky(Sigma)  

    best_x = None
    best_f = np.inf
    for _ in range(100):
        v = z_val + L @ np.random.randn(n)
        x_try = sign_pm1(v)
        f_try = obj_ls(A, b, x_try)
        if f_try < best_f:
            best_f = f_try
            best_x = x_try
    x_d, f_d = best_x, best_f

    # save for printing
    results.append((s, f_a, f_b, f_c, f_d))

# results table
print("Boolean LS heuristics comparison (objective = ||Ax-b||_2^2)")
print("m=50, n=40, A fixed with seed=0; xhat fixed with seed=0; b = A xhat + s*noise")
print("")
print(f"{'s':>4} | {'(i) sign(x_ls)':>16} | {'(ii) sign(z)':>12} | {'(iii) rank1':>11} | {'(iv) 100 samples':>16}")
print("-" * 72)
for s, f_a, f_b, f_c, f_d in results:
    print(f"{s:>4.1f} | {f_a:>16.6f} | {f_b:>12.6f} | {f_c:>11.6f} | {f_d:>16.6f}")

Boolean LS heuristics comparison (objective = ||Ax-b||_2^2)
m=50, n=40, A fixed with seed=0; xhat fixed with seed=0; b = A xhat + s*noise

   s |   (i) sign(x_ls) | (ii) sign(z) | (iii) rank1 | (iv) 100 samples
------------------------------------------------------------------------
 0.5 |        13.422064 |    13.422064 |   13.422064 |        13.422064
 1.0 |       175.810763 |    53.134401 |   53.134401 |        53.134401
 2.0 |       596.639099 |   198.211474 |  198.211474 |       198.211474
 3.0 |      1281.757640 |   389.095376 |  389.095376 |       389.095376
